***Tâche DS-04***

Chargement des données

In [1]:
import pandas as pd

df = pd.read_csv("demande_logements_simulees.csv")
df1 = pd.read_csv("users_simules.csv")

print("Demandes :", df.shape)
print("Users    :", df1.shape)

Demandes : (250, 12)
Users    : (105, 8)


Prévisualisation du contenu des dataframe

In [2]:
df.head()

,demande_id,reference_dossier,user_id,type_logement,statut_actuel,date_soumission,statut_professionnel,situation_matrimoniale,revenu_mensuel,commune,province,region
0,1,ANK-2026-0001,90,Économique,active,2026-01-11 11:52:51,Chômeur/Chômeuse,divorce,416824,Ziniaré,Oubritenga,Plateau-Central
1,2,ANK-2026-0002,49,Sociale,rejetee,2026-03-26 16:17:45,Chômeur/Chômeuse,Divorcé(e),469897,Ouagadougou,Kadiogo,Centre
2,3,ANK-2026-0003,5,Économique,validee,2026-03-25 08:40:02,Chômeuse,Célibataire,52208,Ouagadougou,Kadiogo,Centre
3,4,ANK-2026-0004,61,Sociale,correction_demandee,2026-01-22 14:18:29,Salarié,Marié(e),526660,Ouagadougou,Kadiogo,Centre
4,5,ANK-2026-0005,26,Économique,validee,2026-03-21 06:47:04,Chômeuse,Célibataire,309876,Koudougou,Boulkiemdé,Centre-Ouest


In [3]:
df1.head()

,user_id,uuid_hash,profession,tranche_revenu,commune,created_at,date_naissance,telephone
0,1,usr_770487,Agent de sécurité,< 50k,Koudougou,2025-01-05 20:33:30,1993-04-06,+226 59228452
1,2,usr_356787,Comptable,50k-150k,Bobo-Dioulasso,2025-01-28 02:17:37,1993-01-13,+226 53439167
2,3,usr_809570,Agent de sécurité,< 50k,Ouagadougou,2025-04-25 15:53:40,1978-12-04,+226 51066449
3,4,usr_131244,Agent de sécurité,< 50k,Ouagadougou,2025-02-12 10:55:36,1970-06-08,+226 66956909
4,5,usr_731262,Comptable,< 50k,Ouagadougou,2025-04-19 23:35:29,1968-12-01,+226 74025631


In [4]:
df.isna().sum()

demande_id                 0
reference_dossier          0
user_id                    0
type_logement              0
statut_actuel              0
date_soumission            0
statut_professionnel       0
situation_matrimoniale     0
revenu_mensuel             0
commune                   19
province                   0
region                     0
dtype: int64

In [5]:
df1.isna().sum()

user_id           0
uuid_hash         0
profession        0
tranche_revenu    0
commune           0
created_at        0
date_naissance    0
telephone         0
dtype: int64

In [6]:
(df == 'undefined').sum()

demande_id                 0
reference_dossier          0
user_id                    0
type_logement              0
statut_actuel              0
date_soumission            0
statut_professionnel       0
situation_matrimoniale     0
revenu_mensuel             0
commune                   15
province                  15
region                    15
dtype: int64

In [7]:
total = len(df)

vides = df.isna().sum()
undefined = (df == 'undefined').sum()

remplissage = pd.DataFrame({
    'vides': vides,
    'undefined': undefined,
})

remplissage['taux_remplissage_%'] = ((total - vides - undefined) / total * 100).round(1)
remplissage

,vides,undefined,taux_remplissage_%
demande_id,0,0,100.0
reference_dossier,0,0,100.0
user_id,0,0,100.0
type_logement,0,0,100.0
statut_actuel,0,0,100.0
date_soumission,0,0,100.0
statut_professionnel,0,0,100.0
situation_matrimoniale,0,0,100.0
revenu_mensuel,0,0,100.0
commune,19,15,86.4


In [8]:
referentiel = {
    'Ouagadougou':    {'province': 'Kadiogo',    'region': 'Centre'},
    'Bobo-Dioulasso': {'province': 'Houet',      'region': 'Hauts-Bassins'},
    'Koudougou':      {'province': 'Boulkiemdé', 'region': 'Centre-Ouest'},
    'Kaya':           {'province': 'Sanmatenga', 'region': 'Centre-Nord'},
    'Ziniaré':        {'province': 'Oubritenga', 'region': 'Plateau-Central'},
}

In [9]:
valides = df[df['commune'].isin(referentiel.keys())].copy()

print("Lignes gardées :", len(valides))

Lignes gardées : 216


In [10]:
#province et la région attendues

valides['province_attendue'] = valides['commune'].map(lambda c: referentiel[c]['province'])
valides['region_attendue']   = valides['commune'].map(lambda c: referentiel[c]['region'])

valides[['commune', 'province', 'province_attendue']].head(10)

,commune,province,province_attendue
0,Ziniaré,Oubritenga,Oubritenga
1,Ouagadougou,Kadiogo,Kadiogo
2,Ouagadougou,Kadiogo,Kadiogo
3,Ouagadougou,Kadiogo,Kadiogo
4,Koudougou,Boulkiemdé,Boulkiemdé
5,Bobo-Dioulasso,Houet,Houet
6,Ouagadougou,Kadiogo,Kadiogo
7,Ouagadougou,Kadiogo,Kadiogo
8,Ouagadougou,Kadiogo,Kadiogo
10,Kaya,Sanmatenga,Sanmatenga


In [11]:
incoherentes = valides[
    (valides['province'] != valides['province_attendue']) |
    (valides['region']   != valides['region_attendue'])
]

print("Incohérences géographiques :", len(incoherentes))
incoherentes[['demande_id', 'commune', 'province', 'province_attendue']]

Incohérences géographiques : 10


,demande_id,commune,province,province_attendue
11,12,Ouagadougou,Houet,Kadiogo
40,41,Ouagadougou,Houet,Kadiogo
42,43,Ouagadougou,Houet,Kadiogo
93,94,Ouagadougou,Houet,Kadiogo
100,101,Ouagadougou,Houet,Kadiogo
106,107,Ouagadougou,Houet,Kadiogo
115,116,Ouagadougou,Houet,Kadiogo
132,133,Ouagadougou,Houet,Kadiogo
172,173,Ouagadougou,Houet,Kadiogo
193,194,Ouagadougou,Houet,Kadiogo


In [12]:
df1['date_naissance'] = pd.to_datetime(df1['date_naissance'], errors='coerce')

In [13]:
#compter les naissances dans le futur
aujourdhui = pd.Timestamp.today()

futur = df1[df1['date_naissance'] > aujourdhui]

print("Naissances dans le futur :", len(futur))
futur[['user_id', 'date_naissance']]

Naissances dans le futur : 4


,user_id,date_naissance
29,30,2090-05-12
40,41,2090-05-12
79,80,2090-05-12
92,93,2090-05-12


In [14]:
#les âges absurdes
age = (aujourdhui - df1['date_naissance']).dt.days / 365.25

trop_vieux = df1[age > 110]

print("Âge > 110 ans :", len(trop_vieux))
trop_vieux[['user_id', 'date_naissance']]

Âge > 110 ans : 2


,user_id,date_naissance
45,46,1850-01-01
93,94,1850-01-01


In [15]:
#convertir les deux colonnes en dates

df['date_soumission'] = pd.to_datetime(df['date_soumission'])
df1['created_at'] = pd.to_datetime(df1['created_at'])

In [16]:
#relier chaque demande à son compte
verif = df.merge(df1[['user_id', 'created_at']], on='user_id')

verif[['demande_id', 'user_id', 'date_soumission', 'created_at']].head()

,demande_id,user_id,date_soumission,created_at
0,1,90,2026-01-11 11:52:51,2025-05-24 22:23:11
1,2,49,2026-03-26 16:17:45,2025-02-14 10:56:45
2,3,5,2026-03-25 08:40:02,2025-04-19 23:35:29
3,4,61,2026-01-22 14:18:29,2025-01-19 02:38:09
4,5,26,2026-03-21 06:47:04,2025-05-26 01:20:12


In [17]:
#trouver les demandes avant la création du compte
avant_compte = verif[verif['date_soumission'] < verif['created_at']]

print("Demandes avant la création du compte :", len(avant_compte))

Demandes avant la création du compte : 0


In [18]:
synthese = pd.DataFrame({
    'Contrôle': [
        'Communes vides',
        "Communes 'undefined'",
        'Incohérences région/province',
        'Naissances dans le futur',
        'Âge > 110 ans',
        'Demandes avant création du compte',
    ],
    'Nombre': [
        df['commune'].isna().sum(),
        (df['commune'] == 'undefined').sum(),
        len(incoherentes),
        len(futur),
        len(trop_vieux),
        len(avant_compte),
    ],
    'Total contrôlé': [
        len(df),
        len(df),
        len(valides),
        len(df1),
        len(df1),
        len(df),
    ],
})

synthese['Taux anomalie %'] = (synthese['Nombre'] / synthese['Total contrôlé'] * 100).round(1)

synthese

,Contrôle,Nombre,Total contrôlé,Taux anomalie %
0,Communes vides,19,250,7.6
1,Communes 'undefined',15,250,6.0
2,Incohérences région/province,10,216,4.6
3,Naissances dans le futur,4,105,3.8
4,Âge > 110 ans,2,105,1.9
5,Demandes avant création du compte,0,250,0.0


***Résume***
- 13,2 % des demandes ont une commune inutilisable (vide ou "undefined")
- 4,6 % des demandes valides ont une province qui ne correspond pas à la commune
- 7 comptes ont une date de naissance impossible
- Aucune demande n'est antérieure à la création du compte

***Tâche DS-05***

In [19]:
colonnes_geo = ['commune', 'province', 'region']

# Une case est "cassée" si elle est vide OU contient 'undefined'
cassees = df[colonnes_geo].isna() | (df[colonnes_geo] == 'undefined')

# Une demande est à corriger si AU MOINS UNE de ses 3 cases est cassée
a_corriger = df[cassees.any(axis=1)]

print("Demandes à corriger :", len(a_corriger))
a_corriger[['demande_id', 'user_id', 'commune', 'province', 'region']].head(10)

Demandes à corriger : 34


,demande_id,user_id,commune,province,region
9,10,25,NaN,Kadiogo,Centre
12,13,36,undefined,undefined,undefined
15,16,69,NaN,Boulkiemdé,Centre-Ouest
26,27,33,undefined,undefined,undefined
29,30,69,undefined,undefined,undefined
47,48,23,undefined,undefined,undefined
50,51,1,NaN,Boulkiemdé,Centre-Ouest
51,52,37,NaN,Kadiogo,Centre
54,55,84,NaN,Houet,Hauts-Bassins
82,83,99,NaN,Kadiogo,Centre


In [20]:
commune_du_compte = df1.set_index('user_id')['commune']

commune_du_compte.head()

user_id
1         Koudougou
2    Bobo-Dioulasso
3       Ouagadougou
4       Ouagadougou
5       Ouagadougou
Name: commune, dtype: str

In [21]:
df_corrige = df.copy()

# 1. Quelles demandes ont une commune cassée ?
commune_cassee = df_corrige['commune'].isna() | (df_corrige['commune'] == 'undefined')
a_remplir = df_corrige[commune_cassee]

# 2. Quelle est la bonne commune ? On la cherche dans le compte.
nouvelles = a_remplir['user_id'].map(commune_du_compte)

# 3. On écrit une ligne de journal pour chaque changement
journal_commune = pd.DataFrame({
    'demande_id': a_remplir['demande_id'],
    'champ': 'commune',
    'ancienne_valeur': a_remplir['commune'],
    'nouvelle_valeur': nouvelles,
})

# 4. On applique la correction
df_corrige.loc[commune_cassee, 'commune'] = nouvelles

print("Communes corrigées :", len(journal_commune))
journal_commune.head(10)

Communes corrigées : 34


,demande_id,champ,ancienne_valeur,nouvelle_valeur
9,10,commune,NaN,Ouagadougou
12,13,commune,undefined,Ouagadougou
15,16,commune,NaN,Koudougou
26,27,commune,undefined,Ziniaré
29,30,commune,undefined,Koudougou
47,48,commune,undefined,Bobo-Dioulasso
50,51,commune,NaN,Koudougou
51,52,commune,NaN,Ouagadougou
54,55,commune,NaN,Bobo-Dioulasso
82,83,commune,NaN,Ouagadougou


In [22]:
print("Communes vides après correction :", df_corrige['commune'].isna().sum())
print("Communes 'undefined' après correction :", (df_corrige['commune'] == 'undefined').sum())

Communes vides après correction : 0
Communes 'undefined' après correction : 0


In [23]:
province_cassee = df_corrige['province'].isna() | (df_corrige['province'] == 'undefined')
a_remplir = df_corrige[province_cassee]

nouvelles = a_remplir['commune'].map(lambda c: referentiel[c]['province'])

journal_province = pd.DataFrame({
    'demande_id': a_remplir['demande_id'],
    'champ': 'province',
    'ancienne_valeur': a_remplir['province'],
    'nouvelle_valeur': nouvelles,
})

df_corrige.loc[province_cassee, 'province'] = nouvelles

print("Provinces corrigées :", len(journal_province))

Provinces corrigées : 15


In [24]:
region_cassee = df_corrige['region'].isna() | (df_corrige['region'] == 'undefined')
a_remplir = df_corrige[region_cassee]

nouvelles = a_remplir['commune'].map(lambda c: referentiel[c]['region'])

journal_region = pd.DataFrame({
    'demande_id': a_remplir['demande_id'],
    'champ': 'region',
    'ancienne_valeur': a_remplir['region'],
    'nouvelle_valeur': nouvelles,
})

df_corrige.loc[region_cassee, 'region'] = nouvelles

print("Régions corrigées :", len(journal_region))

Régions corrigées : 15


In [25]:
journal = pd.concat([journal_commune, journal_province, journal_region], ignore_index=True)

print("Total des modifications :", len(journal))
journal.groupby('champ').size()

Total des modifications : 64


champ
commune     34
province    15
region      15
dtype: int64

In [26]:
ref = pd.read_csv("ref_demandes_propres.csv")
ref.shape

(250, 12)

In [27]:

comparaison = df_corrige.merge(
    ref[['demande_id', 'commune', 'province', 'region']],
    on='demande_id',
    suffixes=('', '_ref')
)

for champ in ['commune', 'province', 'region']:
    ecarts = (comparaison[champ] != comparaison[champ + '_ref']).sum()
    print(champ, ": écarts avec la référence =", ecarts)

commune : écarts avec la référence = 0
province : écarts avec la référence = 10
region : écarts avec la référence = 0


In [28]:
journal.to_csv("journal_modifications_DS05.csv", index=False, encoding="utf-8")
df_corrige.to_csv("demandes_corrigees.csv", index=False, encoding="utf-8")

print("Journal :", len(journal), "lignes")
print("Demandes corrigées :", len(df_corrige), "lignes")

Journal : 64 lignes
Demandes corrigées : 250 lignes


***Tâche DS-06***

In [29]:
#trouver les téléphones qui apparaissent plusieurs fois

doublons_tel = df1[df1.duplicated(subset='telephone', keep=False)]

print("Lignes concernées :", len(doublons_tel))
print("Numéros partagés :", doublons_tel['telephone'].nunique())

doublons_tel.sort_values('telephone')[['user_id', 'telephone', 'date_naissance', 'commune']]

Lignes concernées : 10
Numéros partagés : 5


,user_id,telephone,date_naissance,commune
47,48,+226 50241284,1969-11-17,Bobo-Dioulasso
104,105,+226 50241284,1969-11-17,Bobo-Dioulasso
56,57,+226 56382351,1968-08-05,Bobo-Dioulasso
102,103,+226 56382351,1968-08-05,Bobo-Dioulasso
95,96,+226 68786573,1994-12-29,Koudougou
100,101,+226 68786573,1994-12-29,Koudougou
64,65,+226 73373732,1978-12-22,Kaya
103,104,+226 73373732,1978-12-22,Kaya
98,99,+226 76918544,1978-06-20,Ouagadougou
101,102,+226 76918544,1978-06-20,Ouagadougou


In [30]:
# Un identifiant de groupe par téléphone
doublons_tel = doublons_tel.sort_values(['telephone', 'user_id']).copy()
doublons_tel['rang'] = doublons_tel.groupby('telephone').cumcount()

premier = doublons_tel[doublons_tel['rang'] == 0]
second  = doublons_tel[doublons_tel['rang'] == 1]

liste_metier = premier.merge(second, on='telephone', suffixes=('_1', '_2'))

liste_metier['critere'] = 'même téléphone'
liste_metier['meme_date_naissance'] = liste_metier['date_naissance_1'] == liste_metier['date_naissance_2']
liste_metier['meme_commune'] = liste_metier['commune_1'] == liste_metier['commune_2']

liste_metier = liste_metier[['telephone', 'user_id_1', 'user_id_2',
                          'meme_date_naissance', 'meme_commune', 'critere']]

print("Paires à vérifier :", len(liste_metier))
liste_metier

Paires à vérifier : 5


,telephone,user_id_1,user_id_2,meme_date_naissance,meme_commune,critere
0,+226 50241284,48,105,True,True,même téléphone
1,+226 56382351,57,103,True,True,même téléphone
2,+226 68786573,96,101,True,True,même téléphone
3,+226 73373732,65,104,True,True,même téléphone
4,+226 76918544,99,102,True,True,même téléphone


In [31]:
aujourdhui = pd.Timestamp.today()

df1_valide = df1[(df1['date_naissance'] <= aujourdhui) & (df1['date_naissance'].dt.year > 1900)]

meme_naissance = df1_valide[df1_valide.duplicated(subset=['date_naissance', 'commune'], keep=False)]

print("Lignes concernées :", len(meme_naissance))
meme_naissance.sort_values(['date_naissance', 'commune'])[['user_id', 'telephone', 'date_naissance', 'commune']]

Lignes concernées : 10


,user_id,telephone,date_naissance,commune
56,57,+226 56382351,1968-08-05,Bobo-Dioulasso
102,103,+226 56382351,1968-08-05,Bobo-Dioulasso
47,48,+226 50241284,1969-11-17,Bobo-Dioulasso
104,105,+226 50241284,1969-11-17,Bobo-Dioulasso
98,99,+226 76918544,1978-06-20,Ouagadougou
101,102,+226 76918544,1978-06-20,Ouagadougou
64,65,+226 73373732,1978-12-22,Kaya
103,104,+226 73373732,1978-12-22,Kaya
95,96,+226 68786573,1994-12-29,Koudougou
100,101,+226 68786573,1994-12-29,Koudougou


***Tâche DS-07***

In [32]:
statuts = df['statut_professionnel'].value_counts().reset_index()
statuts.columns = ['valeur_brute', 'nombre']
statuts

,valeur_brute,nombre
0,Chômeur/Chômeuse,39
1,Indépendant(e),26
2,Salarié(e),25
3,Indépendant,16
4,Chômeuse,15
5,independant,15
6,Fonctionnaire,15
7,Agent de l'État,14
8,fonctionnaire,13
9,Chômeur,13


In [33]:
correspondance_statut = {
    # -> Chômeur/Chômeuse
    'Chômeur':              'Chômeur/Chômeuse',
    'chomeur':              'Chômeur/Chômeuse',
    'Chômeuse':             'Chômeur/Chômeuse',
    # -> Indépendant(e)
    'Indépendant':          'Indépendant(e)',
    'Indépendante':         'Indépendant(e)',
    'independant':          'Indépendant(e)',
    # -> Fonctionnaire
    'fonctionnaire':        'Fonctionnaire',
    "Fonctionnaire d'État": 'Fonctionnaire',
    "Agent de l'État":      'Fonctionnaire',
    # -> Salarié(e)
    'Salarié':              'Salarié(e)',
    'Salariée':             'Salarié(e)',
    'salarie':              'Salarié(e)',
}

statuts['valeur_normalisee'] = statuts['valeur_brute'].replace(correspondance_statut)
statuts

,valeur_brute,nombre,valeur_normalisee
0,Chômeur/Chômeuse,39,Chômeur/Chômeuse
1,Indépendant(e),26,Indépendant(e)
2,Salarié(e),25,Salarié(e)
3,Indépendant,16,Indépendant(e)
4,Chômeuse,15,Chômeur/Chômeuse
5,independant,15,Indépendant(e)
6,Fonctionnaire,15,Fonctionnaire
7,Agent de l'État,14,Fonctionnaire
8,fonctionnaire,13,Fonctionnaire
9,Chômeur,13,Chômeur/Chômeuse


In [34]:
df['statut_normalise'] = df['statut_professionnel'].replace(correspondance_statut)

print("Valeurs différentes :", df['statut_normalise'].nunique())
df['statut_normalise'].value_counts()

Valeurs différentes : 4


statut_normalise
Chômeur/Chômeuse    79
Indépendant(e)      68
Fonctionnaire       54
Salarié(e)          49
Name: count, dtype: int64

In [35]:
ref = pd.read_csv("ref_demandes_propres.csv")

comp = df.merge(ref[['demande_id', 'statut_professionnel']],
                on='demande_id', suffixes=('', '_ref'))

ecarts = (comp['statut_normalise'] != comp['statut_professionnel_ref']).sum()
print("Écarts avec la référence :", ecarts)

Écarts avec la référence : 0


In [36]:
statuts.to_csv("correspondance_statut_professionnel.csv", index=False, encoding="utf-8")

In [37]:
situations_inv = df['situation_matrimoniale'].value_counts().reset_index()
situations_inv.columns = ['valeur_brute', 'nombre']
situations_inv

,valeur_brute,nombre
0,Marié(e),35
1,Célibataire,33
2,Divorcé(e),29
3,Veuf/Veuve,27
4,celibataire,18
5,Célibataire,17
6,Mariée,17
7,Veuve,14
8,Divorcée,14
9,divorce,11


In [38]:
correspondance_situation = {
    # -> Célibataire
    'celibataire':  'Célibataire',
    'Célibataire ': 'Célibataire',   # avec un espace à la fin
    # -> Marié(e)
    'marie':        'Marié(e)',
    'Marié':        'Marié(e)',
    'Mariée':       'Marié(e)',
    # -> Divorcé(e)
    'divorce':      'Divorcé(e)',
    'Divorcé':      'Divorcé(e)',
    'Divorcée':     'Divorcé(e)',
    # -> Veuf/Veuve
    'veuf':         'Veuf/Veuve',
    'Veuve':        'Veuf/Veuve',
}

situations_inv['valeur_normalisee'] = situations_inv['valeur_brute'].replace(correspondance_situation)
situations_inv

,valeur_brute,nombre,valeur_normalisee
0,Marié(e),35,Marié(e)
1,Célibataire,33,Célibataire
2,Divorcé(e),29,Divorcé(e)
3,Veuf/Veuve,27,Veuf/Veuve
4,celibataire,18,Célibataire
5,Célibataire,17,Célibataire
6,Mariée,17,Marié(e)
7,Veuve,14,Veuf/Veuve
8,Divorcée,14,Divorcé(e)
9,divorce,11,Divorcé(e)


In [39]:
df['situation_normalisee'] = df['situation_matrimoniale'].replace(correspondance_situation)

print("Valeurs différentes :", df['situation_normalisee'].nunique())
df['situation_normalisee'].value_counts()

Valeurs différentes : 4


situation_normalisee
Célibataire    68
Marié(e)       67
Divorcé(e)     64
Veuf/Veuve     51
Name: count, dtype: int64

In [40]:
comp = df.merge(ref[['demande_id', 'situation_matrimoniale']],
                on='demande_id', suffixes=('', '_ref'))

ecarts = (comp['situation_normalisee'] != comp['situation_matrimoniale_ref']).sum()
print("Écarts avec la référence :", ecarts)

Écarts avec la référence : 0


In [41]:
situations_inv.to_csv("correspondance_situation_matrimoniale.csv", index=False, encoding="utf-8")

In [42]:
professions_inv = df1['profession'].value_counts().reset_index()
professions_inv.columns = ['valeur_brute', 'nombre']
professions_inv

,valeur_brute,nombre
0,Comptable,18
1,Agent de sécurité,15
2,Entrepreneur indépendant,10
3,Fonctionnaire d'État,10
4,Artisan (Menuisier/Maçon),10
5,Commerçant,8
6,Chauffeur,8
7,Secrétaire de direction,7
8,Infirmier / Agent de santé,4
9,Prof,4


In [43]:
print(df1['profession'].map(repr).value_counts())

profession
'Comptable'                     18
'Agent de sécurité'             15
'Entrepreneur indépendant'      10
"Fonctionnaire d'État"          10
'Artisan (Menuisier/Maçon)'     10
'Commerçant'                     8
'Chauffeur'                      8
'Secrétaire de direction'        7
'Infirmier / Agent de santé'     4
'Prof'                           4
'Chauffeur routier'              3
'Instituteur / Enseignant'       3
'COMMERÇANT'                     1
'chauffeur '                     1
'instituteur'                    1
'Enseignant'                     1
'commercant'                     1
Name: count, dtype: int64


In [44]:
correspondance_profession = {
    # -> Instituteur / Enseignant
    'Enseignant':        'Instituteur / Enseignant',
    'instituteur':       'Instituteur / Enseignant',
    'Prof':              'Instituteur / Enseignant',
    # -> Commerçant
    'commercant':        'Commerçant',
    'COMMERÇANT':        'Commerçant',
    'Commerce':          'Commerçant',
    # -> Chauffeur
    'chauffeur ':        'Chauffeur',    
    'Chauffeur routier': 'Chauffeur',
}

professions_inv['valeur_normalisee'] = professions_inv['valeur_brute'].replace(correspondance_profession)
professions_inv


,valeur_brute,nombre,valeur_normalisee
0,Comptable,18,Comptable
1,Agent de sécurité,15,Agent de sécurité
2,Entrepreneur indépendant,10,Entrepreneur indépendant
3,Fonctionnaire d'État,10,Fonctionnaire d'État
4,Artisan (Menuisier/Maçon),10,Artisan (Menuisier/Maçon)
5,Commerçant,8,Commerçant
6,Chauffeur,8,Chauffeur
7,Secrétaire de direction,7,Secrétaire de direction
8,Infirmier / Agent de santé,4,Infirmier / Agent de santé
9,Prof,4,Instituteur / Enseignant


In [45]:
df1['profession_normalisee'] = df1['profession'].replace(correspondance_profession)

print("Valeurs différentes :", df1['profession_normalisee'].nunique())
df1['profession_normalisee'].value_counts()

Valeurs différentes : 10


profession_normalisee
Comptable                     18
Agent de sécurité             15
Chauffeur                     12
Entrepreneur indépendant      10
Fonctionnaire d'État          10
Artisan (Menuisier/Maçon)     10
Commerçant                    10
Instituteur / Enseignant       9
Secrétaire de direction        7
Infirmier / Agent de santé     4
Name: count, dtype: int64

In [46]:
ref_users = pd.read_csv("ref_users_propres.csv")

comp = df1.merge(ref_users[['user_id', 'profession']],
                 on='user_id', suffixes=('', '_ref'))

print("Lignes comparées :", len(comp))
ecarts = (comp['profession_normalisee'] != comp['profession_ref']).sum()
print("Écarts avec la référence :", ecarts)

Lignes comparées : 100
Écarts avec la référence : 0


In [47]:
professions_inv.to_csv("correspondance_profession.csv", index=False, encoding="utf-8")

In [48]:
df['revenu_mensuel'].describe()

count       250.000000
mean     313633.036000
std      173009.069636
min       24402.000000
25%      153835.500000
50%      301796.000000
75%      481713.250000
max      599524.000000
Name: revenu_mensuel, dtype: float64

In [49]:
bornes = [0, 50000, 150000, 300000, float('inf')]
noms   = ['Moins de 50 000', '50 000 à 150 000', '150 000 à 300 000', '300 000 et plus']

table_tranches = pd.DataFrame({
    'tranche': noms,
    'minimum': bornes[:-1],
    'maximum': bornes[1:],
})
table_tranches

,tranche,minimum,maximum
0,Moins de 50 000,0,50000.0
1,50 000 à 150 000,50000,150000.0
2,150 000 à 300 000,150000,300000.0
3,300 000 et plus,300000,inf


In [50]:
df['tranche_revenu_normalisee'] = pd.cut(
    df['revenu_mensuel'],
    bins=bornes,
    labels=noms,
    right=False
)

df[['demande_id', 'revenu_mensuel', 'tranche_revenu_normalisee']].head()

,demande_id,revenu_mensuel,tranche_revenu_normalisee
0,1,416824,300 000 et plus
1,2,469897,300 000 et plus
2,3,52208,50 000 à 150 000
3,4,526660,300 000 et plus
4,5,309876,300 000 et plus


In [51]:
print("Revenus sans tranche :", df['tranche_revenu_normalisee'].isna().sum())
df['tranche_revenu_normalisee'].value_counts().sort_index()

Revenus sans tranche : 0


tranche_revenu_normalisee
Moins de 50 000        9
50 000 à 150 000      50
150 000 à 300 000     66
300 000 et plus      125
Name: count, dtype: int64

In [52]:
test = pd.DataFrame({'revenu_mensuel': [49999, 50000, 149999, 150000, 300000]})
test['tranche'] = pd.cut(test['revenu_mensuel'], bins=bornes, labels=noms, right=False)
test

,revenu_mensuel,tranche
0,49999,Moins de 50 000
1,50000,50 000 à 150 000
2,149999,50 000 à 150 000
3,150000,150 000 à 300 000
4,300000,300 000 et plus


In [53]:
table_tranches.to_csv("tranches_revenu_mensuel.csv", index=False, encoding="utf-8")